# Accessibility Audit Pipeline — End-to-End Walkthrough

This notebook walks through the full **two-pass accessibility audit pipeline**, from
raw HTML to LLM-ready prompts. It uses the trimmed Vision Aid homepage as a test file.

The pipeline has five stages:

| Stage | What happens | Cost |
|-------|-------------|------|
| **Pass 1** — Programmatic Checks | Rule-based detection across 3 checkers (43 rules) | Free |
| **Pass 1.5** — Build Filters | Programmatic findings exclude items from LLM payloads | Free |
| **Pass 2a** — Extract | Three extractors parse HTML into structured JSON payloads | Free |
| **Pass 2b** — Slice + Fill | Payloads sliced per prompt, inserted into templates | Free |
| **Pass 2c** — Call LLM | Each filled prompt sent to Anthropic API | ~$0.05–$0.32 |

**Design principle**: Pass 1 results **filter** Pass 2 inputs. Items that failed a binary
check (no alt attribute, no label, broken reference) are excluded from LLM evaluation —
there is no value in judging the quality of something that doesn't exist.

This notebook does **not** make live API calls. It demonstrates the pipeline up to the
point of calling the LLM, then points to `run_pipeline.py` for actual execution.

---
## Setup

Clone the repository and install dependencies (required on Google Colab).
If running locally from the project root, skip the clone cell.

In [ ]:
# Clone the repo and install dependencies (required on Google Colab)
# If running locally from the project root, skip this cell.

import os

if os.getenv("COLAB_RELEASE_TAG"):
    repo_dir = "visionaid-a11y-llm-audit"
    branch = "feat/llm-prompt-pipeline"
    if os.path.isdir(repo_dir):
        # Already cloned from a previous run — pull latest
        %cd {repo_dir}
        !git fetch origin {branch}
        !git checkout {branch}
        !git reset --hard origin/{branch}
        print(f"\nUpdated existing clone to latest {branch}.")
    else:
        !git clone -b {branch} https://github.com/marimdz/visionaid-a11y-llm-audit.git
        %cd {repo_dir}
    !pip install -e . -q
    print("\nSetup complete — dependencies installed.")
else:
    print("Not running on Colab — skipping clone. Make sure you're in the project root.")

In [ ]:
import sys
import json
from pathlib import Path
from collections import Counter

# Resolve project root (this notebook lives at the project root)
PROJECT_ROOT = Path(".").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

HTML_FILE = PROJECT_ROOT / "test_files" / "dat_visionaid_home.html"


def estimate_tokens(text: str) -> int:
    """Rough token estimate: ~4 characters per token for English/HTML."""
    return max(1, len(text) // 4)


print(f"Project root : {PROJECT_ROOT}")
print(f"HTML file    : {HTML_FILE.name}  ({HTML_FILE.stat().st_size / 1024:.0f} KB)")
print(f"File exists  : {HTML_FILE.exists()}")

In [ ]:
# How big is the raw HTML?
raw_text = HTML_FILE.read_text(encoding="utf-8", errors="replace")
raw_bytes = HTML_FILE.stat().st_size
raw_tokens = estimate_tokens(raw_text)

print(f"File size   : {raw_bytes:>10,} bytes  ({raw_bytes / 1024:.0f} KB)")
print(f"Characters  : {len(raw_text):>10,}")
print(f"Est. tokens : {raw_tokens:>10,}")
print()
print(f"At Claude Sonnet pricing ($3/million input tokens):")
print(f"  Sending this raw HTML once = ${raw_tokens * 3 / 1_000_000:.4f}")
print()
print("The extractors will reduce this dramatically by keeping only")
print("the semantically relevant elements (headings, links, forms, images, etc.).")

---
## Pass 1 — Programmatic Binary Checks

**Run first.** These checks are objectively verifiable directly from the HTML — no LLM
interpretation needed. Three checkers cover different WCAG areas:

| Checker | Script | Rules | Focus |
|---------|--------|-------|-------|
| **CL01** Semantic | `programmatic/semantic_checklist_01.py` | 29 | Page title, headings, landmarks, links, iframes, tables |
| **CL02** Forms | `programmatic/forms_checklist_02.py` | 7 | Labels, fieldsets, required fields, aria-describedby |
| **CL03** Non-text | `programmatic/nontext_checklist_03.py` | 7 | Image alt, SVG, canvas, media, objects |

Pass 1 produces two outputs:
1. **Direct findings** — binary violations reported immediately
2. **Filter sets** — used to exclude already-caught items from Pass 2

In [ ]:
from processing_scripts.programmatic.semantic_checklist_01 import audit_html_file
from processing_scripts.programmatic.forms_checklist_02 import audit_forms
from processing_scripts.programmatic.nontext_checklist_03 import audit_nontext

sem_findings = audit_html_file(str(HTML_FILE))
form_findings = audit_forms(str(HTML_FILE))
ntext_findings = audit_nontext(str(HTML_FILE))

all_prog = sem_findings + form_findings + ntext_findings

checkers = [
    ("CL01 Semantic", sem_findings),
    ("CL02 Forms", form_findings),
    ("CL03 Non-text", ntext_findings),
]

LINE = "\u2500"

print("=" * 60)
print("  PASS 1 \u2014 PROGRAMMATIC AUDIT RESULTS")
print("=" * 60)

for name, issues in checkers:
    print(f"\n  {name}  ({len(issues)} issue(s))")
    print(f"  {LINE * 46}")
    if not issues:
        print("    No issues found.")
    else:
        by_rule = {}
        for i in issues:
            key = i["rule_id"]
            by_rule.setdefault(key, (i["rule_name"], 0))
            by_rule[key] = (by_rule[key][0], by_rule[key][1] + 1)
        for rule_id in sorted(by_rule):
            rule_name, count = by_rule[rule_id]
            suffix = f"  x {count}" if count > 1 else ""
            print(f"    [{rule_id}] {rule_name}{suffix}")

print(f"\n  {LINE * 56}")
print(f"  Total Pass 1 issues : {len(all_prog)}")

In [ ]:
# Sample findings (first 5)
print("Sample programmatic findings (first 5):\n")
for f in all_prog[:5]:
    print(f"  [{f['rule_id']}] {f['rule_name']}")
    print(f"    {f['description']}")
    print()

---
## Pass 1.5 — Build Filter Sets

When a rule fires, it means something is missing or broken. There is nothing for the
LLM to evaluate quality of — so those items are excluded from Pass 2.

| Rule that fires | Excluded from Pass 2 |
|---|---|
| `PAGE_TITLE_001` / `_003` (missing/empty title) | CL01 `page_title` prompt skipped entirely |
| `HEAD_004` (empty heading) | That heading filtered from CL01 `heading_structure` |
| `IFRAME_001` / `_002` (missing/empty iframe title) | Those iframes filtered from CL01 `iframe_titles` |
| `FORM_LABEL_001` (no label at all) | That field filtered from CL02 `label_quality` |
| `NON_TEXT_002` (actionable image, no/empty alt) | That image filtered from CL03 `actionable_image_alt` |

In [ ]:
from processing_scripts.llm.filters import build_filter_flags

filter_flags = build_filter_flags(sem_findings, form_findings, ntext_findings)

labels = {
    "skip_title_prompt":  "Skip CL01 page_title prompt",
    "head_empty_fired":   "Filter empty headings from CL01 heading_structure",
    "iframe_rules_fired": "Filter untitled iframes from CL01 iframe_titles",
    "form_label_fired":   "Filter unlabelled fields from CL02 label_quality",
    "ntext_action_fired": "Filter no-alt actionable images from CL03",
}

LINE = "\u2500"

print("Pass 1 -> Pass 2 filter summary")
print(f"  {'Filter':<55} {'Active?':>8}")
print(f"  {LINE * 65}")
for key, label in labels.items():
    active = "YES" if filter_flags.get(key) else "no"
    print(f"  {label:<55} {active:>8}")
print()
if filter_flags["skip_prompts"]:
    print(f"  Prompts skipped entirely: {filter_flags['skip_prompts']}")
else:
    print("  No prompts skipped entirely.")
print("  Items marked 'no' pass through to LLM quality evaluation unchanged.")

---
## Pass 2a — Extract Structured Payloads

Three extractors parse HTML into structured JSON payloads, discarding all CSS,
JavaScript, and layout wrappers — keeping only what the LLM needs to evaluate
WCAG compliance.

| Extractor | Focus Area | Key Output Fields |
|-----------|-----------|-------------------|
| **CL01** | Semantic Structure | `page_title`, `headings`, `flagged_links`, `landmarks`, `tables`, `iframes` |
| **CL02** | Forms | `forms[].fields` (label_source, effective_label, instructions), `forms[].groups` |
| **CL03** | Non-text Content | `images` (informative/decorative/actionable/complex), `svgs`, `icon_fonts`, `media` |

In [ ]:
from processing_scripts.llm_preprocessing.semantic_checklist_01 import extract as cl01_extract
from processing_scripts.llm_preprocessing.forms_checklist_02 import extract as cl02_extract
from processing_scripts.llm_preprocessing.nontext_checklist_03 import extract as cl03_extract

cl01_payload = cl01_extract(str(HTML_FILE))
cl02_payload = cl02_extract(str(HTML_FILE))
cl03_payload = cl03_extract(str(HTML_FILE))

cl01_json = json.dumps(cl01_payload)
cl02_json = json.dumps(cl02_payload)
cl03_json = json.dumps(cl03_payload)

cl01_tokens = estimate_tokens(cl01_json)
cl02_tokens = estimate_tokens(cl02_json)
cl03_tokens = estimate_tokens(cl03_json)
combined_tokens = cl01_tokens + cl02_tokens + cl03_tokens

print(f"{'Checklist':<25} {'~Tokens':>8}  {'% of raw':>8}")
print(f"{'-' * 45}")
print(f"{'CL01 - Semantic':<25} {cl01_tokens:>8,}  {cl01_tokens / raw_tokens * 100:>7.1f}%")
print(f"{'CL02 - Forms':<25} {cl02_tokens:>8,}  {cl02_tokens / raw_tokens * 100:>7.1f}%")
print(f"{'CL03 - Non-text':<25} {cl03_tokens:>8,}  {cl03_tokens / raw_tokens * 100:>7.1f}%")
print(f"{'-' * 45}")
print(f"{'Combined':<25} {combined_tokens:>8,}  {combined_tokens / raw_tokens * 100:>7.1f}%")
print(f"{'Raw HTML':<25} {raw_tokens:>8,}  {'100.0%':>8}")
print()
print(f"Token reduction: {(1 - combined_tokens / raw_tokens) * 100:.0f}%")

In [ ]:
# CL01 payload summary
print("CL01 Payload — Section summary")
print(f"{'Section':<22} {'Items':>6}")
print(f"{'-' * 32}")
for key, val in cl01_payload.items():
    if isinstance(val, list):
        print(f"{key:<22} {len(val):>6}")
    elif isinstance(val, dict) and key == "images":
        total = sum(len(v) for v in val.values())
        print(f"{key:<22} {total:>6}  (subcategories: {list(val.keys())})")
    elif isinstance(val, dict):
        print(f"{key:<22} {'(dict)':>6}")
    else:
        print(f"{key:<22} {repr(val)[:30]:>30}")

In [ ]:
# CL02 payload summary
forms = cl02_payload["forms"]
all_fields = [f for form in forms for f in form["fields"]]
all_groups = [g for form in forms for g in form["groups"]]
required_fields = [f for f in all_fields if f.get("required")]
src_counts = Counter(f["label_source"] for f in all_fields)

print(f"CL02 — Forms summary")
print(f"  Total forms    : {len(forms)}")
print(f"  Total fields   : {len(all_fields)}")
print(f"  Total groups   : {len(all_groups)}")
print(f"  Required fields: {len(required_fields)}")
print()
if all_fields:
    print("  Label source breakdown:")
    for src, count in src_counts.most_common():
        flag = "  <-- problem" if src in ("placeholder_only", "none") else ""
        print(f"    {src:<22} {count:>4}{flag}")
else:
    print("  (no form fields found)")

In [ ]:
# CL03 payload summary
img = cl03_payload["images"]
total_images = sum(len(v) for v in img.values())

print(f"CL03 — Non-text content summary")
print(f"  Images — informative : {len(img['informative'])}")
print(f"  Images — decorative  : {len(img['decorative'])}")
print(f"  Images — actionable  : {len(img['actionable'])}  (inside <a> or <button>)")
print(f"  Images — complex     : {len(img['complex'])}")
print(f"  Total images         : {total_images}")
print(f"  SVGs (non-hidden)    : {len(cl03_payload['svgs'])}")
print(f"  Icon fonts           : {len(cl03_payload['icon_fonts'])}")
print(f"  Media elements       : {len(cl03_payload['media'])}")

---
## Pass 2a.5 — Apply Filters to Payloads

Before slicing, we apply the Pass 1 filter flags to remove items that already have
programmatic findings. This happens *before* slicing, so slicers naturally produce
smaller output and empty-slice detection works correctly.

In [ ]:
from processing_scripts.llm.filters import (
    apply_cl01_filters,
    apply_cl02_filters,
    apply_cl03_filters,
)

cl01_filtered = apply_cl01_filters(cl01_payload, filter_flags)
cl02_filtered = apply_cl02_filters(cl02_payload, filter_flags)
cl03_filtered = apply_cl03_filters(cl03_payload, filter_flags)

payloads = {"CL01": cl01_filtered, "CL02": cl02_filtered, "CL03": cl03_filtered}

# Show what changed
diffs = [
    ("CL01 headings", len(cl01_payload.get("headings", [])), len(cl01_filtered.get("headings", []))),
    ("CL01 iframes", len(cl01_payload.get("iframes", [])), len(cl01_filtered.get("iframes", []))),
    ("CL03 actionable images", len(cl03_payload["images"]["actionable"]), len(cl03_filtered["images"]["actionable"])),
]

print("Filter effects on payloads:")
print(f"  {'Section':<28} {'Before':>8} {'After':>8} {'Removed':>8}")
print(f"  {'-' * 56}")
for label, before, after in diffs:
    removed = before - after
    marker = f"  ({removed} filtered)" if removed > 0 else ""
    print(f"  {label:<28} {before:>8} {after:>8} {removed:>8}{marker}")

---
## Pass 2b — The Prompt Registry

The prompt registry (`processing_scripts/llm/registry.py`) is the single source of truth
for what the pipeline evaluates. It defines 21 `PromptSpec` entries, each mapping a name
to its checklist, prompt template, slicer function, and WCAG criteria.

In [ ]:
from processing_scripts.llm.registry import PROMPT_REGISTRY

print(f"Total prompts in registry: {len(PROMPT_REGISTRY)}")
print()
print(f"{'Name':<30} {'CL':>3}  {'WCAG':<20} {'Type':<8} {'Summary?'}")
print(f"{'-' * 80}")
for spec in PROMPT_REGISTRY:
    wcag = ", ".join(spec.wcag_criteria)
    summary = "Yes" if spec.is_summary else ""
    print(f"{spec.name:<30} {spec.checklist:>3}  {wcag:<20} {spec.output_type:<8} {summary}")

In [ ]:
# Slice all payloads and show token budget per prompt (post-filtering)
from processing_scripts.llm.slicers import get_slicer, is_empty_slice

print(f"{'Prompt':<30} {'~Tokens':>8}  {'Status':>12}")
print(f"{'-' * 55}")

total_slice_tokens = 0
non_empty_count = 0

for spec in PROMPT_REGISTRY:
    if spec.is_summary:
        print(f"{spec.name:<30} {'--':>8}  {'(summary)':>12}")
        continue

    if spec.name in filter_flags["skip_prompts"]:
        print(f"{spec.name:<30} {'--':>8}  {'FILTERED':>12}")
        continue

    slicer_fn = get_slicer(spec.payload_slicer)
    payload_json = slicer_fn(payloads[spec.checklist])
    tokens = estimate_tokens(payload_json)
    empty = is_empty_slice(payload_json)

    status = "SKIP (empty)" if empty else ""
    print(f"{spec.name:<30} {tokens:>8,}  {status:>12}")

    if not empty:
        total_slice_tokens += tokens
        non_empty_count += 1

print(f"{'-' * 55}")
print(f"Non-empty prompts: {non_empty_count}")
print(f"Total payload tokens (non-empty): ~{total_slice_tokens:,}")

In [ ]:
# Slice detail: page_title
page_title_slicer = get_slicer("slice_page_title")
page_title_json = page_title_slicer(payloads["CL01"])

print("page_title slice (what the LLM receives):")
print(page_title_json)
print()

# Slice detail: flagged_links
link_slicer = get_slicer("slice_flagged_links")
links_json = link_slicer(payloads["CL01"])
links_data = json.loads(links_json)

print(f"flagged_links slice: {len(links_data)} links, ~{estimate_tokens(links_json):,} tokens")
print()
if links_data:
    print("First 5 flagged links:")
    for link in links_data[:5]:
        aria = link.get("aria_label") or ""
        aria_str = f"  aria_label={repr(aria)}" if aria else ""
        print(f"  text={repr(link.get('text', '')):<30}{aria_str}")

---
## Pass 2b — Fill Prompt Templates

Prompt templates live in `.txt` files under `processing_scripts/llm/`. Each file
contains multiple prompts separated by dashed headers. The `templates.py` module
parses these files, finds the right prompt by index, and replaces `{payload}` with
the sliced JSON.

In [ ]:
from processing_scripts.llm.templates import fill_template, get_template

# Show a raw template before filling
page_title_spec = PROMPT_REGISTRY[0]  # page_title
raw_template = get_template(page_title_spec)

print(f"Prompt: {page_title_spec.name}")
print(f"File:   {page_title_spec.prompt_file}")
print(f"Index:  {page_title_spec.prompt_index}")
print(f"WCAG:   {page_title_spec.wcag_criteria}")
print()
print("--- Raw template (before payload substitution) ---")
print(raw_template)

In [ ]:
# Fill all non-empty, non-summary prompts and show token costs
filled_prompts = {}  # name -> (spec, filled_text)
skipped_prompts = []

for spec in PROMPT_REGISTRY:
    if spec.is_summary:
        skipped_prompts.append((spec.name, "summary (optional)"))
        continue

    if spec.name in filter_flags["skip_prompts"]:
        skipped_prompts.append((spec.name, "Pass 1 filter"))
        continue

    slicer_fn = get_slicer(spec.payload_slicer)
    payload_json = slicer_fn(payloads[spec.checklist])

    if spec.skip_if_empty and is_empty_slice(payload_json):
        skipped_prompts.append((spec.name, "empty payload"))
        continue

    filled_text = fill_template(spec, payload_json)
    filled_prompts[spec.name] = (spec, filled_text)

total_prompt_tokens = 0
print(f"{'Prompt':<30} {'~Tokens':>8}  {'WCAG':<20}")
print(f"{'-' * 62}")
for name, (spec, text) in filled_prompts.items():
    tokens = estimate_tokens(text)
    total_prompt_tokens += tokens
    wcag = ", ".join(spec.wcag_criteria)
    print(f"{name:<30} {tokens:>8,}  {wcag:<20}")

print(f"{'-' * 62}")
print(f"{'TOTAL':<30} {total_prompt_tokens:>8,}")
print(f"{'API calls needed':<30} {len(filled_prompts):>8}")
print()
sonnet_cost = total_prompt_tokens * 3 / 1_000_000
print(f"Estimated input cost (Sonnet @ $3/MTok): ${sonnet_cost:.4f}")
print()
if skipped_prompts:
    print(f"Skipped ({len(skipped_prompts)}):")
    for name, reason in skipped_prompts:
        print(f"  {name}: {reason}")

In [ ]:
# Preview one filled prompt: heading_structure
if "heading_structure" in filled_prompts:
    spec, text = filled_prompts["heading_structure"]
    print(f"Prompt: {spec.name}")
    print(f"WCAG:   {spec.wcag_criteria}")
    print(f"Tokens: ~{estimate_tokens(text):,}")
    print()
    preview = text[:2000]
    print(preview)
    if len(text) > 2000:
        print(f"\n... ({len(text) - 2000} more characters) ...")
else:
    print("heading_structure prompt was skipped (empty payload).")

In [ ]:
# Complete token budget summary
print("=" * 56)
print("  PIPELINE TOKEN SUMMARY")
print("=" * 56)
print(f"  Raw HTML file              : ~{raw_tokens:>8,} tokens")
print(f"  After extraction (3 JSONs) : ~{combined_tokens:>8,} tokens  ({combined_tokens / raw_tokens * 100:.1f}% of raw)")
print(f"  After slicing (prompts)    : ~{total_prompt_tokens:>8,} tokens  ({total_prompt_tokens / raw_tokens * 100:.1f}% of raw)")
print()
print(f"  Prompts to send            : {len(filled_prompts):>8}")
print(f"  Prompts skipped            : {len(skipped_prompts):>8}")
print(f"  Programmatic findings      : {len(all_prog):>8}  (free)")
print()
print(f"  Estimated cost (Sonnet)    : ${total_prompt_tokens * 3 / 1_000_000:.4f} input")
print(f"  vs. sending raw HTML       : ${raw_tokens * 3 / 1_000_000:.4f} input")
print(f"  Savings                    : {(1 - total_prompt_tokens / raw_tokens) * 100:.0f}%")

---
## Architecture

```
HTML file (raw)
    |
    |  PASS 1 -- Programmatic (run first, milliseconds)
    |
    +-- programmatic/semantic_checklist_01.py   --> 29 rules --> binary findings
    +-- programmatic/forms_checklist_02.py      -->  7 rules --> binary findings
    +-- programmatic/nontext_checklist_03.py    -->  7 rules --> binary findings
                                                          |
                                                          |  filter: items that failed
                                                          |  a binary check excluded
                                                          v
    |  PASS 2 -- LLM Quality Evaluation (on filtered payloads)
    |
    +-- llm_preprocessing/*.py  --> extract structured JSON payloads
    +-- llm/filters.py         --> apply Pass 1 filters to payloads
    +-- llm/slicers.py         --> slice payloads per prompt
    +-- llm/templates.py       --> fill {payload} in prompt templates
    +-- run_pipeline.py        --> send to Anthropic API  --> up to 21 LLM calls
                                                                      |
                                                                      v
                                                         merge all findings
                                                                      |
                                                                      v
                                                         generate_report.py --> CSV
```

---
## CLI Reference: `run_pipeline.py`

Everything above is automated by `entry_points/run_pipeline.py`.

### Dry run (no API calls, no cost)

```bash
python entry_points/run_pipeline.py --html test_files/dat_visionaid_home.html --dry-run
```

### Live run (calls the Anthropic API)

```bash
python entry_points/run_pipeline.py --html test_files/dat_visionaid_home.html
```

### Generate a CSV report

```bash
python entry_points/generate_report.py
```

### Pipeline options

| Flag | Default | Description |
|------|---------|-------------|
| `--html` | (required) | Path to the HTML file to analyze |
| `--output-dir` | `./output` | Directory for results |
| `--model` | `claude-sonnet-4-20250514` | Anthropic model to use |
| `--dry-run` | off | Generate prompts without calling the API |
| `--include-summaries` | off | Include the 3 cross-cutting summary prompts |
| `--show-cost` | off | Print estimated dollar cost after the run |

### Output structure

```
output/
+-- manifest.json                 # Run metadata, filter info, token counts
+-- programmatic_findings.json    # All 3 programmatic checkers combined
+-- payloads/                     # Raw extractor output
|   +-- cl01_payload.json
|   +-- cl02_payload.json
|   +-- cl03_payload.json
+-- prompts/                      # One file per prompt
    +-- page_title.json
    +-- heading_structure.json
    +-- ...
```